# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The Contract:


*   **Unit of analysis**: One row represents exactly one `report_date` × `client_hash_id` × `content_hash_id`.   

*   **Tables used**: `fact_content_daily_performance_sample` (for safe iteration) joined with `dim_content`.  

*   **Time window**: A single mid-panel month partition: `2025-02-01` through `2025-02-28`.  (not a mid-panel month but will be used as an example as dataset is too large to run through)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



*   **Context:** `content_hash_id`, `client_hash_id`, `report_date`. Used exclusively for grouping, joining, and splitting, never for the model to learn from.

*   **Label / Proxy**: `ctr` (Click-Through Rate). This is the observed outcome we are analyzing to see which signals drive clicks.

*   **Excluded**: `trend_pct` and `trend_direction`. Excluded because they are derived metrics that create a "label trap" (data leakage).


*   **Features:**

1.   `word_count`: Knowable at the decision moment because it is a static property of the published text.

2.   `content_type`: Knowable at the decision moment because the format is defined at publication.

3. `has_video` (derived missingness flag): Knowable at the decision moment because it reflects the structural layout of the page.

4. `gsc_avg_position_prev30`: Knowable at the decision moment because historical search ranks from the prior window are already logged.

5. `ga4_traffic_prev30`: Knowable at the decision moment because past traffic volumes are fully observed before the current prediction window.






## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
# Load the dataset using data from the month of February as it is faster to load
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

print("Connecting to stream...")
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=hf_token
)

print("Streaming data to find February 2025 records...")
records = []

for i, row in enumerate(ds):
    date_str = str(row.get('report_date', ''))

    if date_str.startswith('2025-02'):
        records.append(row)

    if len(records) >= 10000:
        break

df_feb = pd.DataFrame(records)

if not df_feb.empty:
    df_feb['report_date'] = pd.to_datetime(df_feb['report_date'])
    print(f"Success! Loaded a sample slice of February 2025 with {len(df_feb)} rows.")
else:
    print("Could not find records.")

Connecting to stream...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Streaming data to find February 2025 records...
Success! Loaded a sample slice of February 2025 with 10000 rows.


In [22]:
# 1. Prove the Grain

grain_check = df_feb.groupby(['report_date', 'client_hash_id', 'content_hash_id']).size().reset_index(name='row_count')


duplicates = grain_check[grain_check['row_count'] > 1]

print(f"Duplicate grain records found: {len(duplicates)}")

Duplicate grain records found: 0


In [23]:
# 2. Prove Row Count and Date Span
total_rows = len(df_feb)

min_date = df_feb['report_date'].min()
max_date = df_feb['report_date'].max()

print(f"Total rows in slice: {total_rows}")
print(f"Date span: {min_date.date()} to {max_date.date()}")

Total rows in slice: 10000
Date span: 2025-02-10 to 2025-02-14


In [24]:
# 3. Prove Availability

available_df = df_feb[df_feb['ga4_data_available'] == True]

surviving_rows = len(available_df)
survival_rate = (surviving_rows / total_rows) * 100

print(f"Rows with GA4 data available: {surviving_rows}")
print(f"Percentage of slice surviving: {survival_rate:.1f}%")

Rows with GA4 data available: 0
Percentage of slice surviving: 0.0%


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



*   **Unbalanced history:** We cannot establish a universal timeline because history depth differs wildly per client, requiring us to check `gsc_data_start` before defining cross-client time windows. Additionally, a third of the clients lack usable analytics history entirely.  

*   **GSC-only early rows:** We cannot assume early zeros in traffic metrics mean "zero engagement." Rows dated before a client's `ga4_data_start` have GA4 columns zero-filled with the `ga4_data_available` flag set to FALSE.  


*   **Window overlaps:** If using aggregated 90-day query tables, the fixed 90-day window overlaps the snapshot's final months, meaning only `*_prev30-style` columns are safe to use as features to prevent leakage.


**Leakage Trap Demonstration:**
This script evaluates if content ranks on Page 1 (gsc_avg_position <= 10). First, we clean missing
values and filter out avg_position = 0 (which means "no data", not rank zero). We then train a
"leaky" model by intentionally including a derived column (gsc_sum_position, which mathematically
leaks the average when combined with impressions) to watch the ROC-AUC score jump artificially high.
Finally, we remove the leaking feature and retrain to reveal the honest, un-leaked baseline score.

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df_clean = df_feb.dropna(subset=['gsc_avg_position', 'gsc_impressions', 'gsc_sum_position']).copy()

df_clean = df_clean[df_clean['gsc_avg_position'] > 0]

y = (df_clean['gsc_avg_position'] <= 10).astype(int)

# The Trap
honest_features = ['gsc_impressions']
features_with_leak = honest_features + ['gsc_sum_position']

X_trap = df_clean[features_with_leak]
X_train_trap, X_test_trap, y_train, y_test = train_test_split(X_trap, y, test_size=0.2, random_state=42)

model_trap = RandomForestClassifier(random_state=42, max_depth=3)
model_trap.fit(X_train_trap, y_train)
leak_probs = model_trap.predict_proba(X_test_trap)[:, 1]

print(f"SCORE WITH LEAKAGE (The Trap): ROC-AUC = {roc_auc_score(y_test, leak_probs):.3f}")
print("This is suspiciously high because 'gsc_sum_position' did the math for the model\n")


# Honest Model
X_honest = df_clean[honest_features]
X_train_h, X_test_h, _, _ = train_test_split(X_honest, y, test_size=0.2, random_state=42)

model_honest = RandomForestClassifier(random_state=42, max_depth=3)
model_honest.fit(X_train_h, y_train)
honest_probs = model_honest.predict_proba(X_test_h)[:, 1]

print(f"HONEST SCORE (No Leakage): ROC-AUC = {roc_auc_score(y_test, honest_probs):.3f}")
print("This lower score reflects a realistic, non-leaky prediction baseline.")

SCORE WITH LEAKAGE (The Trap): ROC-AUC = 0.966
This is suspiciously high because 'gsc_sum_position' did the math for the model

HONEST SCORE (No Leakage): ROC-AUC = 0.590
This lower score reflects a realistic, non-leaky prediction baseline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.